# Script for semi-supervised keyword expansion

## Importing dependencies and loading existing preprocessed data

In [ ]:
# importing dependencies and keywords
from dependencies import *
from keywords import *

In [ ]:
import fasttext
import fasttext.util

In [ ]:
# loading existing preprocessed data
df = pd.read_csv(preprocessed_csv_filepath, encoding='utf-8-sig')
print(f"Loaded {len(df)} rows")

## FastText - expanding keywords with new words outside corpus

In [ ]:
# loading pretrained FastText model (may take some time)
ft = fasttext.load_model(os.path.join(os.path.expanduser("~"), "Downloads", "cc.en.300.bin"))
print(f"Model loaded — vocabulary size: {ft.get_words().__len__()}")

In [ ]:
# defining already used keywords to ensure that suggestions are not duplicates of these
existing_keywords = {kw.lower() for kw_set in product_quality_feature_keywords.values() for kw in kw_set}

# getting 30 similar words per product quality feature
n_similar = 30

print("Beginning FastText Keyword Expansion...")
for category, keywords in product_quality_feature_keywords.items():
    print(f"\n{'='*60}\n{category}\n{'='*60}")

    all_suggestions = {}
    for kw in keywords:
        # skipping currency symbols as these can cause errors
        if kw in ('$', '£', '€'):
            continue
        # convert bigrams to underscore format for FastText lookup
        kw_ft = kw.lower().replace(' ', '_')
        neighbours = ft.get_nearest_neighbors(kw_ft, k=n_similar)
        for score, word in neighbours:
            if word.lower() not in all_suggestions:
                all_suggestions[word.lower()] = score

    new_suggestions = {}
    # filtering out irrelevant suggestions, e.g., stopwords
    for word, score in all_suggestions.items():
    # allow unigrams and bigrams
        parts = word.split('_')
        if not all(p.isalpha() for p in parts):
            continue
        if len(parts) > 2:
            continue
        if len(word.replace('_', '')) <= 2:
            continue
        if word in nlp.Defaults.stop_words:
            continue

        # lemmatize potential new keywords and compare them against existing keywords (which are already lemmas)
        candidate_lemmas = {token.lemma_ for token in nlp(word.replace('_', ' '))}
        if candidate_lemmas & existing_keywords:
            continue
        new_suggestions[word] = score

    # providing 15 new suggestions
    new_suggestions = sorted(new_suggestions.items(), key=lambda x: x[1], reverse=True)

    print(f"Top new potential keywords:")
    for word, score in new_suggestions[:15]:
        print(f"{word}: {score}")

## Anchored CorEx - statistically verifying + extending existing topics and keywords

In [ ]:
# only looking at longer documents in the corpus with 20+ characters
df_corex = df[df["preprocessed_text_corex"].astype(str).str.len() >= 20].copy()
print(f"Documents: {len(df_corex)}")

In [ ]:
# creating vectorizer with english stopwords
vectorizer = CountVectorizer(
    stop_words="english",
    min_df=5,
    max_df=0.90,
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b",
    ngram_range=(1, 2))

dtm = vectorizer.fit_transform(df_corex["preprocessed_text_corex"].astype(str))
vocab = vectorizer.get_feature_names_out()
print(f"Vocabulary size: {len(vocab)}")

In [ ]:
# auto-deriving anchor words from the product quality feature dictionaries
anchor_words = []
anchor_labels = []
for category, keywords in product_quality_feature_keywords.items():
    # only use keywords that survived the vectorizer vocabulary
    vocab_set = set(vocab)
    valid_anchors = [kw for kw in keywords if kw in vocab_set and len(kw) > 2 and kw != "$"]
    if valid_anchors:
        anchor_words.append(valid_anchors[:8])  # cap at 8 anchors per topic
        anchor_labels.append(category.replace("_keywords", ""))

print(f"Anchor categories: {anchor_labels}")
for label, anchors in zip(anchor_labels, anchor_words):
    print(f"  {label:<25}: {anchors}")
n_free_topics  = 8   # free topics for unknown categories
n_total_topics = len(anchor_words) + n_free_topics

In [ ]:
# training and fitting CorEx
print(f"Training CorEx ({n_total_topics} topics, {len(anchor_words)} anchored + {n_free_topics} free)...")

corex_topic_model = ct.Corex(
    n_hidden=n_total_topics,
    words=vocab,
    max_iter=500,
    seed=42,
    verbose=True)

print(f"Fitting CorEx...")
corex_topic_model.fit(
    dtm,
    words=vocab,
    anchors=anchor_words,
    anchor_strength=3)

print("Done")

In [ ]:
# printing topics and their top words
topic_labels = anchor_labels + [f"Free_{i+1}" for i in range(n_free_topics)]
n_top_words = 20
print("CorEx Topics:")
for i, topic_words in enumerate(corex_topic_model.get_topics(n_words=n_top_words)):
    words = [w for w, *_ in topic_words]
    print(f"\n{topic_labels[i]}:")
    print("  " + ", ".join(words))

In [ ]:
# printing most representative documents per topic
print("Most representative document per topic:")
doc_topic_matrix = corex_topic_model.transform(dtm)
raw_documents = df_corex["text"].astype(str).tolist()

for topic_id in range(n_total_topics):
    top_doc_idx = doc_topic_matrix[:, topic_id].argmax()
    print(f"\n{topic_labels[topic_id]}:")
    print(f"{raw_documents[top_doc_idx][:300]}")

## Guided BERTopic - semantically verifying + extending existing topics and keywords

In [ ]:
# only looking at longer documents in the corpus with 20+ characters
df_bert = df[df["preprocessed_text_bert"].astype(str).str.len() >= 20].copy()
print(f"Documents: {len(df_bert)}")

# defining which column to use for embeddings (minimal processing) and which to use for vectorizer (more aggressive processing, which was applied to the CorEx)
embedding_documents = df_bert["preprocessed_text_bert"].astype(str).tolist()
vectorizer_documents = df_bert["preprocessed_text_corex"].astype(str).tolist()

In [ ]:
# set up vectorizer
vectorizer = CountVectorizer(
    stop_words="english",
    min_df=0.001,
    max_df=0.90,
    ngram_range=(1, 2),
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b",)
vectorizer.fit(vectorizer_documents)
vocab_set = set(vectorizer.get_feature_names_out())

In [ ]:
# auto-deriving seed words from the product quality feature dictionaries
seed_topic_list  = []
seed_topic_labels = []
for category, keywords in product_quality_feature_keywords.items():
    valid_seeds = [
        kw for kw in keywords
        if kw in vocab_set and len(kw) > 2 and kw not in ('$', '£', '€')]
    if valid_seeds:
        seed_topic_list.append(valid_seeds[:8])
        seed_topic_labels.append(category.replace("_keywords", ""))

print(f"Seed categories ({len(seed_topic_labels)}):")
for label, seeds in zip(seed_topic_labels, seed_topic_list):
    print(f"{label}: {seeds}")

In [ ]:
# loading sentencetransformer
print("Loading sentence transformer...")
embedding_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2", device="cuda")

In [ ]:
# training guided BERTopic
print("Training guided BERTopic...")

representation_model = KeyBERTInspired()

topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer,
    seed_topic_list=seed_topic_list,
    representation_model = representation_model,
    nr_topics="auto",
    min_topic_size=25,
    verbose=True)

# precompute embeddings separately 
embeddings = embedding_model.encode(embedding_documents, show_progress_bar=True)
# pass lemmatized preprocessed text for topic word extraction, but raw embeddings for clustering
topics, probs = topic_model.fit_transform(vectorizer_documents, embeddings=embeddings)
print("Training complete")

In [ ]:
# show topics and keywords
print(f"Found {len(topic_model.get_topic_info())} topics")

print("BERTopic topics:")
for topic_id in sorted(topic_model.get_topic_info()["Topic"].tolist()):
    if topic_id == -1:
        continue
    topic_words = topic_model.get_topic(topic_id)
    words       = [w for w, _ in topic_words[:20]]
    print(f"\nTopic {topic_id}:")
    print("  " + ", ".join(words))